# RAG Benchmark — Generation Stage on Colab

Runs the optional generation and answer-scoring stages of the COMP702 RAG
benchmark on a Colab GPU. Retrieval is already complete and is **not** repeated
here: every model answers the byte-identical context that the retrieval
benchmark froze, so differences between models are attributable to the model.

**Repository:** https://github.com/mahesh062003/ai-benchmark

## Before you start

Your Google Drive folder `AI Benchmark` must contain:

| Path | What | Size |
|---|---|---|
| `AI Benchmark/datasets/` | the six raw datasets | ~1 GB |
| `AI Benchmark/artifacts/corpora/` | chunk texts used to build prompts | ~352 MB |
| `AI Benchmark/artifacts/benchmark.sqlite` | the completed retrieval run | ~291 MB |

`artifacts/indexes/` is **not** needed — retrieval has already run.

Upload `artifacts/corpora/` and `artifacts/benchmark.sqlite` from
`C:\AI BENCHMARK\artifacts\` if they are not in Drive yet.

## Why this notebook does not write SQLite directly to Drive

Google Drive is mounted through FUSE, which does not implement the file locking
SQLite relies on. Writing a database there during a long run risks
`database is locked` errors and, in the worst case, a corrupted file holding
every result you have.

So the notebook copies the database to Colab's local disk, runs against it
there, and **copies it back to Drive every 10 minutes** using SQLite's online
backup API, which is safe against a database being written to. Nothing is lost
if the session dies: the Drive copy is at most 10 minutes behind, and the run
resumes from whatever it contains.

## Runtime

Use an **L4** if you have one. Roughly 15 hours for 7,200 generations plus
scoring, which exceeds Colab's 12-hour cap, so expect two sessions. Re-running
this notebook resumes rather than restarting.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

# 7-9B models at fp16 need roughly 18 GB. On a 16 GB card Ollama will fall back
# to partial CPU offload, which still works but is several times slower.

## 1b · Keep the session alive

Colab disconnects a browser it considers idle after roughly ninety minutes,
**regardless of what is still computing**. That, not the twelve-hour cap, is
what ends most long unattended runs: the GPU is working, nobody is clicking,
and the session is dropped anyway.

Run the cell below, then also paste the printed snippet into the browser
console (**F12 → Console**). The console copy is the one that reliably works,
because a notebook output frame is sandboxed away from the page chrome that
owns the connect button.

Neither survives a **closed tab or a sleeping machine**. Leave the tab open,
the lid up, and the laptop on mains power.

In [ ]:
KEEPALIVE_JS = """
setInterval(() => {
  const b = document.querySelector("colab-connect-button");
  if (b && b.shadowRoot) {
    b.shadowRoot.querySelector("#connect")?.click();
    console.log("ping", new Date().toLocaleTimeString());
  }
}, 60000);
"""

try:
    from google.colab import output
    output.eval_js(KEEPALIVE_JS)
    print('keep-alive registered from the notebook (best effort)\n')
except Exception as exc:
    print(f'could not register from the notebook: {exc}\n')

print('Paste this into the browser console (F12 -> Console) as well:')
print('-' * 70)
print(KEEPALIVE_JS.strip())
print('-' * 70)
print('You should then see a "ping" line every minute in the console.')
print('If you do not, it is not running and the session will idle out.')

## 2 · Mount Drive and verify the layout

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/AI Benchmark')
DRIVE_DATASETS = DRIVE / 'datasets'
DRIVE_ARTIFACTS = DRIVE / 'artifacts'

print(f'Drive folder: {DRIVE}')
if not DRIVE.exists():
    raise SystemExit(
        f'{DRIVE} not found. Check the folder name is exactly "AI Benchmark" '
        'and that it sits at the top level of My Drive.'
    )

required = {
    'datasets/':                DRIVE_DATASETS,
    'artifacts/corpora/':       DRIVE_ARTIFACTS / 'corpora',
    'artifacts/benchmark.sqlite': DRIVE_ARTIFACTS / 'benchmark.sqlite',
}
missing = []
for label, path in required.items():
    ok = path.exists()
    print(f'  {"OK  " if ok else "MISS"}  {label}')
    if not ok:
        missing.append(label)
if missing:
    raise SystemExit(
        'Missing from Drive: ' + ', '.join(missing) +
        '\nUpload them from C:\\AI BENCHMARK\\ before continuing.'
    )

## 3 · Clone the repository

In [ ]:
import subprocess
from pathlib import Path

REPO = 'https://github.com/mahesh062003/ai-benchmark.git'
PROJECT = Path('/content/ai-benchmark')

if PROJECT.exists():
    print('updating existing clone')
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(PROJECT)], check=True)

%cd /content/ai-benchmark
!git log --oneline -1

## 4 · Install Python dependencies

In [ ]:
# Colab already ships torch, numpy and pandas; pip will leave those alone.
!pip install -q -r /content/ai-benchmark/requirements.txt 2>&1 | tail -5
print('dependencies ready')

## 5 · Install Ollama and pull the four models

About **19 GB** of model weights. They are stored on Colab's local disk, not
Drive, because loading a model over FUSE is far slower than re-downloading it.
Expect 10–20 minutes on the first run of each session.

In [ ]:
import os
import subprocess
import time

import requests

os.environ['OLLAMA_MODELS'] = '/content/ollama_models'
os.makedirs('/content/ollama_models', exist_ok=True)

if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
    # Ollama now ships its release as .tar.zst. Colab has no zstd binary, so the
    # official installer fails at the extraction step and exits 1.
    print('installing zstd, then ollama...')
    subprocess.run('apt-get -qq update && apt-get -qq install -y zstd', shell=True)
    if subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True).returncode != 0:
        print('installer failed; falling back to the release binary')
        subprocess.run(
            'curl -L --retry 3 -o /tmp/ollama.tar.zst '
            'https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst',
            shell=True, check=True,
        )
        subprocess.run(
            'tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C /usr/local',
            shell=True, check=True,
        )
else:
    print('ollama already installed')

# Start the server detached; it must outlive this cell.
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ},
)

for attempt in range(60):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).ok:
            print('ollama server is up')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise SystemExit('ollama did not start; re-run this cell')

MODELS = ['llama3.1', 'gemma2', 'mistral', 'qwen2.5']
for model in MODELS:
    print(f'--- pulling {model} ---')
    subprocess.run(['ollama', 'pull', model], check=True)

!ollama list

## 6 · Stage artifacts onto local disk

Copied from Drive so the database is written on a real filesystem. Only the
corpora and the database are needed; the FAISS indexes are not.

In [ ]:
import shutil
from pathlib import Path

LOCAL_ARTIFACTS = Path('/content/artifacts')
LOCAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
(LOCAL_ARTIFACTS / 'results').mkdir(exist_ok=True)

local_db = LOCAL_ARTIFACTS / 'benchmark.sqlite'
local_corpora = LOCAL_ARTIFACTS / 'corpora'

if not local_corpora.exists():
    print('copying corpora from Drive (a few minutes)...')
    shutil.copytree(DRIVE_ARTIFACTS / 'corpora', local_corpora)
    print('  done')
else:
    print('corpora already staged')

# Always take the newest database so a resumed session continues the same run.
print('copying database from Drive...')
shutil.copy2(DRIVE_ARTIFACTS / 'benchmark.sqlite', local_db)

# Restore the frozen task sets. Without these a resumed session re-runs
# retrieval to rebuild them, which costs about fifteen minutes on MedQA alone.
drive_results = DRIVE_ARTIFACTS / 'results'
restored = 0
if drive_results.exists():
    for item in drive_results.glob('generation_tasks_*.json'):
        shutil.copy2(item, LOCAL_ARTIFACTS / 'results' / item.name)
        restored += 1
print(f'restored {restored} frozen task set(s)')

import sqlite3
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    runs = c.execute('SELECT run_id FROM runs ORDER BY created_at DESC').fetchall()
    agg = c.execute('SELECT COUNT(*) FROM aggregate_metrics').fetchone()[0]
    gens = c.execute('SELECT COUNT(*) FROM generations').fetchone()[0]
print(f'  runs={[r[0] for r in runs]}  aggregate_metrics={agg}  generations={gens}')
if gens:
    print(f'  resuming: {gens} answers already generated will be skipped')

## 7 · Point the framework at these directories

In [ ]:
import os

# Datasets stay on Drive: they are read rarely and never written.
os.environ['RAGBENCH_DATASETS_DIR'] = str(DRIVE_DATASETS)
# Artifacts live on local disk while running, and are synced back to Drive.
os.environ['RAGBENCH_ARTIFACTS_DIR'] = str(LOCAL_ARTIFACTS)

!python -m cli datasets 2>&1 | head -12
print()
print('datasets dir :', os.environ['RAGBENCH_DATASETS_DIR'])
print('artifacts dir:', os.environ['RAGBENCH_ARTIFACTS_DIR'])

## 8 · Start the automatic Drive sync

Copies the database to Drive every 10 minutes using SQLite's online backup,
which is safe while the database is being written. Leave this running.

In [ ]:
import sqlite3
import threading
import time
from datetime import datetime

SYNC_SECONDS = 600
_stop_sync = threading.Event()


def sync_to_drive(reason='periodic'):
    # Back the live database up to Drive without interrupting writers, and carry
    # the frozen task sets across so a resumed session does not rebuild them.
    try:
        source = sqlite3.connect(f'file:{local_db}?mode=ro', uri=True)
        target = sqlite3.connect(str(DRIVE_ARTIFACTS / 'benchmark.sqlite'))
        with target:
            source.backup(target)
        source.close()
        target.close()

        import shutil
        drive_results = DRIVE_ARTIFACTS / 'results'
        drive_results.mkdir(parents=True, exist_ok=True)
        for item in (LOCAL_ARTIFACTS / 'results').glob('generation_tasks_*.json'):
            shutil.copy2(item, drive_results / item.name)

        stamp = datetime.now().strftime('%H:%M:%S')
        print(f'[{stamp}] synced to Drive ({reason})')
    except Exception as exc:                     # never kill the run over a sync
        print(f'sync failed ({exc}); the local database is still intact')


def _loop():
    while not _stop_sync.wait(SYNC_SECONDS):
        sync_to_drive()


threading.Thread(target=_loop, daemon=True).start()
print(f'auto-sync every {SYNC_SECONDS // 60} minutes -> {DRIVE_ARTIFACTS / "benchmark.sqlite"}')

## 9 · Generate

100 questions per dataset x 6 datasets x 3 strategies x 4 models = **7,200
answers**. Models run one at a time to avoid reloading, and every answer is
committed as it is produced, so an interrupted run resumes rather than
restarting.

In [ ]:
import re
import subprocess

# Resume is keyed on run_id: a new id would regenerate all 7,200 answers from
# scratch. The id is recorded on Drive as soon as it appears in the output, so a
# session that dies minutes later can still be continued.
RUN_FILE = DRIVE_ARTIFACTS / 'generation_run_id.txt'
EXPECTED = 7200          # 100 queries x 6 datasets x 3 strategies x 4 models

# Freezing the task sets re-runs retrieval and costs about twenty minutes on
# MedQA alone, so skip the whole stage when every answer already exists.
import sqlite3 as _sq
with _sq.connect(f'file:{local_db}?mode=ro', uri=True) as _c:
    _done = _c.execute('SELECT COUNT(*) FROM generations WHERE answer IS NOT NULL').fetchone()[0]
if _done >= EXPECTED:
    print(f'generation already complete ({_done} answers) -- skipping to scoring')
    raise SystemExit(0)

command = ['python', '-m', 'cli', 'generate-all', '--all',
           '--models', 'llama3.1,gemma2,mistral,qwen2.5', '--limit', '100', '--verbose']

if RUN_FILE.exists():
    run_id = RUN_FILE.read_text().strip()
    print(f'RESUMING run {run_id} -- answers already stored will be skipped\n')
    command += ['--run', run_id]
else:
    print('starting a NEW generation run\n')

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
    if not RUN_FILE.exists():
        found = re.search(r'(genall-[0-9a-f]+)', line)
        if found:
            RUN_FILE.write_text(found.group(1))
            print(f'\n>>> run id saved to Drive: {found.group(1)}\n')
process.wait()

In [ ]:
sync_to_drive('after generation')

import sqlite3
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    print('answers generated:', c.execute('SELECT COUNT(*) FROM generations').fetchone()[0])
    for row in c.execute(
        'SELECT model, COUNT(*), SUM(answer IS NULL) FROM generations GROUP BY model'
    ):
        print(f'  {row[0]:12s} {row[1]:6d} answers, {row[2] or 0} empty')

## 10 · Score the answers

RAGAS faithfulness with a fixed judge (`mistral`, set in `config/default.yaml`)
so every model is rated by the same rater, plus NLI hallucination detection.
Runs after generation and never during it.

In [ ]:
import subprocess
import time

import requests

# The Ollama server can die between generation and scoring. Left unchecked,
# score-answers logs "skipping faithfulness" and exits successfully, so the run
# looks finished while half the measurement is missing. Restart it first.
def ollama_ready(timeout_seconds=120):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            if requests.get('http://localhost:11434/api/tags', timeout=2).ok:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False


if not ollama_ready(timeout_seconds=4):
    print('ollama is down; restarting it before scoring')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, env={**os.environ})
    if not ollama_ready():
        raise SystemExit('ollama would not start -- re-run this cell')
print('ollama is up; starting scoring\n')

In [ ]:
# --config is not optional here. Without it the CLI falls back to the dataclass
# defaults, where faithfulness_sample is None (judge every one of the 7,200
# answers rather than the stratified 1,440) and the judge is the single-model
# default rather than the fixed judge the methodology specifies. The run still
# looks correct while doing five times the work with the wrong rater.
#
# Run with ! rather than subprocess so the progress bars stream into the cell.
# A silent long-running job is indistinguishable from a hung one.
!python -m cli score-answers --config config/default.yaml --verbose

In [ ]:
# Confirm both halves actually ran. Faithfulness is the one that silently skips.
import sqlite3

with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    faith = c.execute(
        'SELECT COUNT(*) FROM generations WHERE faithfulness IS NOT NULL').fetchone()[0]
    halluc = c.execute(
        'SELECT COUNT(*) FROM generations WHERE hallucination IS NOT NULL').fetchone()[0]

print(f'faithfulness scored : {faith}')
print(f'hallucination scored: {halluc}')
sync_to_drive('after scoring')

if faith == 0:
    raise SystemExit(
        'FAITHFULNESS DID NOT RUN. Check the ollama messages above and re-run '
        'the scoring cell before continuing -- do not run the final cell yet, '
        'it stops the Drive backup.'
    )

In [ ]:
!python -m cli results --config config/default.yaml

## 11 · Final sync and export

Run this only once scoring has finished. It stops the periodic Drive backup, so
any work done after it is unprotected until the next manual sync.

In [ ]:
import shutil
import sqlite3
import subprocess

# Refuse to stop the backup while a measurement is still missing: that
# combination is what loses a long unattended run.
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    faith = c.execute(
        'SELECT COUNT(*) FROM generations WHERE faithfulness IS NOT NULL').fetchone()[0]
if faith == 0:
    raise SystemExit(
        'Faithfulness has not been scored. Re-run the scoring cell first; '
        'stopping the Drive backup now would leave a later run unprotected.'
    )

_stop_sync.set()
sync_to_drive('final')

# Results CSVs are small; copy the whole results directory across.
drive_results = DRIVE_ARTIFACTS / 'results'
drive_results.mkdir(parents=True, exist_ok=True)
for item in (LOCAL_ARTIFACTS / 'results').glob('*'):
    if item.is_file():
        shutil.copy2(item, drive_results / item.name)
        print('copied', item.name)

!python -m cli export --config config/default.yaml

# Significance tests retrieval, so it needs the retrieval run. Left to itself it
# picks the most recent run, which by now is a generation run carrying no
# per-query retrieval metrics, and reports "nothing to test".
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    retrieval_run = c.execute(
        "SELECT run_id FROM runs WHERE stage LIKE 'retrieval%'"
        " ORDER BY created_at DESC LIMIT 1"
    ).fetchone()

if retrieval_run:
    print(f'\ntesting significance on retrieval run {retrieval_run[0]}')
    subprocess.run(
        ['python', '-m', 'cli', 'significance',
         '--config', 'config/default.yaml', '--run', retrieval_run[0]],
        check=False,
    )
else:
    print('no retrieval run found; skipping significance')

for item in (LOCAL_ARTIFACTS / 'results').glob('*.csv'):
    shutil.copy2(item, drive_results / item.name)

print()
print('Everything is on Drive. Download artifacts/benchmark.sqlite to your')
print('laptop, drop it into C:\\AI BENCHMARK\\artifacts\\, and run:')
print('    streamlit run dashboard/app.py')

## Resuming after a disconnect

Colab caps sessions at 12 hours, and this run is longer than that. To resume:

1. Reconnect and **run every cell from the top**.
2. Step 6 pulls the partially-filled database back from Drive.
3. `generate-all` skips answers that already exist and continues.

Nothing is regenerated, so a second session costs only the work that remains.

## If the GPU runs out of memory

`gemma2` is the largest model here. On a 16 GB card, replace it with a
quantised tag and record the change as a limitation, since it breaks the
equal-precision control across models:

```python
!ollama pull gemma2:9b-instruct-q4_0
```

## Watching compute units

Units are consumed for every hour the runtime is **connected**, not every hour
the GPU is busy. Disconnect the runtime as soon as the final sync finishes.